# Анатомія памʼяті в Python

У цьому ноутбуці ми подивимось, як Python працює з обʼєктами в памʼяті: посилання, лічильник посилань, цикли, garbage collector, інтернування, розміри обʼєктів і базові інструменти аналізу.


## Обʼєкти, ідентичність і `id`

Кожне значення в Python є обʼєктом. `id()` показує ідентичність обʼєкта під час виконання програми: фактично це спосіб побачити, чи дві змінні вказують на той самий обʼєкт.


In [ ]:
value = 10
value_id = id(value)

print("id(value):", value_id)

del value

try:
    print(value)
except NameError as error:
    print("Після del змінна недоступна:", error)

print("id(10):", id(10))
print("Той самий обʼєкт для малого int:", value_id == id(10))


Тут важливо розділяти змінну й обʼєкт. `del value` прибирає імʼя, але не означає, що саме значення `10` зникло з інтерпретатора. Малі цілі числа Python зазвичай перевикористовує.


## Лічильник посилань

CPython зберігає для кожного обʼєкта кількість активних посилань на нього. `sys.getrefcount()` показує це число, але сам виклик функції теж тимчасово додає одне посилання.


In [ ]:
import sys

print("'new object':", sys.getrefcount("new object"))
print("0:", sys.getrefcount(0))

number = 0
message = "new object"

print("'new object' після змінної:", sys.getrefcount("new object"))
print("0 після змінної:", sys.getrefcount(0))


Точні числа можуть відрізнятися між версіями Python і навіть між середовищами запуску. Нас цікавить не абсолютне значення, а те, що кількість посилань змінюється, коли зʼявляються нові імена або передавання у функцію. Тут, як бачимо по результатах, кількість посилань не змінилась, причина цьому кешування в CPython на малі незмінювані обʼєкти, тож тестувати це треба не на таких типах.


In [ ]:
import sys


def show_refcount(obj: object) -> None:
    print(f"Всередині функції: {sys.getrefcount(obj)}")


items = [1, 2, 3]
print(f"Після створення: {sys.getrefcount(items)}")

alias = items
print(f"Після еліаса: {sys.getrefcount(items)}")

show_refcount(items)
print(f"Після виклику функції: {sys.getrefcount(items)}")

del alias
print(f"Після видалення еліаса: {sys.getrefcount(items)}")


`alias` не створює копію списку. Це ще одне імʼя для того самого обʼєкта, тому лічильник посилань збільшується. Коли імʼя видаляється, лічильник зменшується.


## Циклічні посилання

Reference counting добре працює для простих випадків, але має слабке місце: обʼєкти можуть посилатися один на одного. Тоді їхній лічильник не стане нулем, навіть якщо з програми до них уже немає доступу.


In [ ]:
import sys

# Обʼєкт посилається сам на себе.
offices = []
offices.append(offices)
print("self-reference:", sys.getrefcount(offices))

# Два обʼєкти посилаються один на одного.
departments = []
companies = []
departments.append(companies)
companies.append(departments)

print("departments:", sys.getrefcount(departments))
print("companies:", sys.getrefcount(companies))


Такі структури часто виникають у графах, деревах із посиланням на батьківський вузол або складних моделях даних. Саме для цього в Python є garbage collector, який шукає ізольовані цикли.


In [ ]:
import gc
import weakref


class Node:
    def __init__(self, name: str) -> None:
        self.name = name
        self.peer = None

    def __repr__(self) -> str:
        return f"Node({self.name})"


gc.disable()

try:
    first = Node("A")
    second = Node("B")

    first.peer = second
    second.peer = first

    first_ref = weakref.ref(first)
    second_ref = weakref.ref(second)

    del first
    del second

    print("Before gc.collect():")
    print("first_ref() ->", first_ref())
    print("second_ref() ->", second_ref())

    collected = gc.collect()

    print("\nAfter gc.collect():")
    print("collected ->", collected)
    print("first_ref() ->", first_ref())
    print("second_ref() ->", second_ref())
finally:
    gc.enable()


`weakref` дає змогу перевірити, чи обʼєкт ще живий, не утримуючи його в памʼяті. До `gc.collect()` цикл існує, після ручного запуску збирача обʼєкти стають недоступними.


## Пороги garbage collector

GC не запускається після кожного створення обʼєкта. Він працює за порогами: Python рахує різницю між створеними й видаленими контейнерними обʼєктами та запускає збір, коли поріг перевищено.


In [ ]:
import gc

original_thresholds = gc.get_threshold()
print("Поточні пороги:", original_thresholds)

gc.set_threshold(500, 10, 10)
print("Нові пороги:", gc.get_threshold())

gc.set_threshold(*original_thresholds)
print("Повернули назад:", gc.get_threshold())


Змінювати ці пороги в реальному коді варто обережно. Але корисно знати, що GC має налаштування і не є “магічним” процесом без правил.


## Інтернування

Python може перевикористовувати деякі обʼєкти: малі числа, ідентифікатори та частину рядків. Для рядків можна явно попросити інтернування через `sys.intern()`.


In [ ]:
import sys

log_levels = ["ERROR", "INFO", "ERROR", "DEBUG", "INFO"]
raw_values = [(" " + level).strip() for level in log_levels]
interned_values = [sys.intern(level) for level in raw_values]

print("Без явного інтернування:")
print("ERROR:", raw_values[0] is raw_values[2])
print("INFO:", raw_values[1] is raw_values[4])

print("\nЗ інтернуванням:")
print("ERROR:", interned_values[0] is interned_values[2])
print("INFO:", interned_values[1] is interned_values[4])


`is` перевіряє не рівність тексту, а чи це той самий обʼєкт. Інтернування корисне для повторюваних коротких рядків, наприклад ключів, статусів або назв подій.


## `sys.getsizeof`: поверхневий розмір

`sys.getsizeof()` показує розмір самого обʼєкта в байтах. Для рядка довший текст зазвичай займає більше памʼяті, але частина розміру - це службові дані самого обʼєкта.


In [ ]:
import sys

texts = [
    "",
    "Hi",
    "Hello, World!",
    "Ukraine is the best country in the world!",
]

for text in texts:
    print(f"{text!r:45} -> {sys.getsizeof(text)} bytes")


Цей розмір не дорівнює просто кількості символів. У кожного Python-обʼєкта є заголовок і метадані, тому навіть порожній рядок має помітний розмір.


## Списки й over-allocation

Список зберігає посилання на елементи та має запас місця для майбутніх `append()`. Тому розмір списку росте не на кожному додаванні, а сходинками.


In [ ]:
import sys

size_five = sys.getsizeof([1, 2, 3, 4, 5])
size_six = sys.getsizeof([1, 2, 3, 4, 5, 6])

print("size for 5 items:", size_five)
print("size for 6 items:", size_six)
print("Однаковий розмір:", size_five == size_six)


In [ ]:
import sys

items = []
previous_size = sys.getsizeof(items)

print(f"len={len(items):3}, size={previous_size}")

for number in range(1, 101):
    items.append(number)
    current_size = sys.getsizeof(items)

    if current_size != previous_size:
        print(f"len={len(items):3}, size={current_size}")
        previous_size = current_size


Це компроміс: Python витрачає трохи зайвої памʼяті, щоб `append()` зазвичай був швидким. Саме тому додавання в список має амортизовану складність `O(1)`.


## Розміри базових контейнерів

Різні контейнери мають різну внутрішню структуру. Список, кортеж, множина і словник зберігають дані по-різному, тому їхні накладні витрати теж різні.


In [ ]:
import sys

samples = [
    [],
    [1],
    [1, 2, 3, 4],
    [1, 2, 3, 4, 5],
    [1, 2, 3, 4, 5, 6],
    ["Ukraine is the best country in the world!"],
    (),
    (1,),
    (1, 2, 3, 4),
    ("Ukraine is the best country in the world!",),
    set(),
    {1},
    {1, 2, 3, 4},
    {},
    {"a": 1},
    {"a": 1, "b": 2, "c": 3},
]

for obj in samples:
    print(f"{repr(obj):55} -> {sys.getsizeof(obj)} bytes")


Зверніть увагу: `getsizeof()` для контейнера не додає розміри всіх вкладених обʼєктів. Він показує лише “обгортку” контейнера і його внутрішню таблицю або масив посилань.


## Поверхневий і глибокий розмір

Для вкладених структур поверхневого розміру замало. Якщо список містить інші списки або рядки, потрібно рекурсивно пройти по обʼєктах і не порахувати один і той самий обʼєкт двічі.


In [ ]:
import sys
from collections.abc import Container, Mapping
from typing import Any


def deep_getsizeof(obj: Any, seen_ids: set[int] | None = None) -> int:
    if seen_ids is None:
        seen_ids = set()

    obj_id = id(obj)
    if obj_id in seen_ids:
        return 0

    seen_ids.add(obj_id)
    size = sys.getsizeof(obj)

    if isinstance(obj, (str, bytes, bytearray)):
        return size

    if isinstance(obj, Mapping):
        return size + sum(
            deep_getsizeof(key, seen_ids) + deep_getsizeof(value, seen_ids)
            for key, value in obj.items()
        )

    if isinstance(obj, Container):
        return size + sum(deep_getsizeof(item, seen_ids) for item in obj)

    return size


example = [[1, 2, 3], 4, 5, "6", "7"]
print("getsizeof for list:", sys.getsizeof(example))
print("deep_getsizeof for list:", deep_getsizeof(example))


In [ ]:
text = "1234567"

print("text:", deep_getsizeof(text))
print("empty list:", deep_getsizeof([]))
print("list with text:", deep_getsizeof([text]))
print("five references to same text:", deep_getsizeof(5 * [text]))


Останній приклад показує важливу деталь: пʼять елементів списку можуть посилатися на один і той самий рядок. Глибокий підрахунок має врахувати рядок один раз, інакше ми перебільшимо споживання памʼяті.


## `tracemalloc`: де росте памʼять

`tracemalloc` відстежує алокації Python-обʼєктів і дає змогу порівнювати знімки памʼяті. Це корисно, коли потрібно знайти рядки коду, після яких памʼяті стало більше.


In [ ]:
import tracemalloc

tracemalloc.start()

numbers = [number for number in range(1_000_000)]
snapshot = tracemalloc.take_snapshot()

for stat in snapshot.statistics("lineno")[:3]:
    print(stat)

current, peak = tracemalloc.get_traced_memory()
print(f"\ncurrent={current / 1024 / 1024:.2f} MB")
print(f"peak={peak / 1024 / 1024:.2f} MB")

tracemalloc.stop()


In [ ]:
import tracemalloc


def build_cache() -> list[str]:
    cache: list[str] = []
    for index in range(50_000):
        cache.append(f"user-event-{index}")
    return cache


tracemalloc.start()
snapshot_before = tracemalloc.take_snapshot()

cache = build_cache()

snapshot_after = tracemalloc.take_snapshot()
top_stats = snapshot_after.compare_to(snapshot_before, "lineno")

print("Топ зростань памʼяті:")
for stat in top_stats[:5]:
    print(stat)

print(f"Cache size: {len(cache)}")
tracemalloc.stop()


У реальному профілюванні важливо дивитися не лише на факт росту, а й на джерело. Якщо певний рядок стабільно додає багато памʼяті, це кандидат для оптимізації або перевірки на витік.


## `memory_profiler` для функцій

`memory_profiler` показує споживання памʼяті по рядках, але це зовнішній пакет. У ноутбуці нижче приклад зроблено безпечним: якщо пакет не встановлений, клітинка просто пояснить це.


In [ ]:
from memory_profiler import profile


@profile
def load_data() -> list[int]:
    data = [number for number in range(1_000_000)]
    return data

loaded_data = load_data()
print(len(loaded_data))


## Підсумок

Python приховує більшість деталей керування памʼяттю, але ці деталі впливають на продуктивність. Коли ми розуміємо посилання, GC, інтернування, over-allocation і вимірювання памʼяті, нам легше писати код, який нормально масштабується.
